# 01: PyTorch Convolutional Neural Networks (CNNs) from Scratch

**Track 09: Computer Vision & Convolutional Neural Networks** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Build and train a 2D Convolutional Neural Network: Conv2d kernels, MaxPooling, Batch Normalization, Dropout regularization, and Cross-Entropy optimization.


## 1. Convolutional Layer Mechanics & Spatial Reduction
2D Convolution formula with kernel $K \in \mathbb{R}^{k \times k}$:
$$(I * K)(i, j) = \sum_{m} \sum_{n} I(i-m, j-n) K(m, n)$$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

class ConvNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        feat = self.features(x)
        flattened = feat.view(feat.size(0), -1)
        out = self.classifier(flattened)
        return out

model = ConvNet(num_classes=10)
print(model)

## 2. Training Loop with Synthetic Digit Batches
Train for 3 epochs with Adam optimizer and CrossEntropyLoss.

In [ ]:
torch.manual_seed(42)
X_dummy = torch.randn(200, 1, 28, 28)
y_dummy = torch.randint(0, 10, (200,))

dataset = TensorDataset(X_dummy, y_dummy)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("=== Starting CNN Training ===")
for epoch in range(3):
    total_loss = 0.0
    correct = 0
    total = 0
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(y_batch)
        _, preds = torch.max(outputs, 1)
        correct += (preds == y_batch).sum().item()
        total += len(y_batch)
        
    epoch_loss = total_loss / total
    epoch_acc = correct / total * 100
    print(f"Epoch {epoch+1}/3 -> Loss: {epoch_loss:.4f} | Batch Accuracy: {epoch_acc:.2f}%")